## Part B – Data Preparation
### B1: Load and Explore Data

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import models, layers

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

amazon_df = pd.read_csv("amazon_cells_labelled.txt", sep="\t", header=None, names=["sentence","label"])
imdb_df = pd.read_csv("imdb_labelled.txt", sep="\t", header=None, names=["sentence","label"])
yelp_df = pd.read_csv("yelp_labelled.txt", sep="\t", header=None, names=["sentence","label"])

data = pd.concat([amazon_df, imdb_df, yelp_df], ignore_index=True)
data.head()

### B1: Check Unusual Characters

In [ ]:
def has_non_ascii(text):
    return any(ord(c) > 127 for c in text)

data["has_non_ascii"] = data["sentence"].apply(has_non_ascii)
data[data["has_non_ascii"]].head()

### B1: Clean Text

In [ ]:
def basic_clean(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

data["clean_sentence"] = data["sentence"].apply(basic_clean)
data[["sentence","clean_sentence"]].head()

### B1: Vocabulary & Length Stats

In [ ]:
eda_tokenizer = Tokenizer(oov_token="<OOV>")
eda_tokenizer.fit_on_texts(data["clean_sentence"])

word_index_full = eda_tokenizer.word_index
vocab_size_full = len(word_index_full) + 1
vocab_size_full

In [ ]:
eda_sequences = eda_tokenizer.texts_to_sequences(data["clean_sentence"])
seq_lengths = [len(s) for s in eda_sequences]

plt.hist(seq_lengths, bins=30)
plt.title("Sequence Length Distribution")
plt.show()

np.percentile(seq_lengths,[90,95,99])

### B2–B5: Tokenization, Padding, Split

In [ ]:
vocab_size = 5000
embedding_dim = 16
max_len = 40

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(data["clean_sentence"])

sequences = tokenizer.texts_to_sequences(data["clean_sentence"])
padded = pad_sequences(sequences, maxlen=max_len, padding="post", truncating="post")
labels = data["label"].values
sample_idx = 0
print("Original cleaned sentence")
print(data['clean_sentence'].iloc[sample_idx])

print("Tokenized sequence")
print(sequences[sample_idx])

print("Padded sequence")
print(padded[sample_idx])

X_temp, X_test, y_temp, y_test = train_test_split(padded, labels, test_size=0.15, stratify=labels, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=SEED)

(len(X_train), len(X_val), len(X_test))

### B6: Save Prepared Data

In [ ]:
pd.DataFrame(X_train).assign(label=y_train).to_csv("prepared_train_data.csv", index=False)
pd.DataFrame(X_val).assign(label=y_val).to_csv("prepared_val_data.csv", index=False)
pd.DataFrame(X_test).assign(label=y_test).to_csv("prepared_test_data.csv", index=False)


## Part C – Network Architecture

In [ ]:
model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_shape=(max_len,),name="embedding"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(1e-3),
    metrics=["accuracy"]
)

model.summary()

## Part D – Training and Evaluation

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)
# number of completed epochs
last_epoch = len(history.history["loss"])

print(f"Final Epoch: {last_epoch}")
print(f"Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Training Loss: {history.history['loss'][-1]:.4f}")
print(f"Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Validation Loss: {history.history['val_loss'][-1]:.4f}")

### D3: Training Curves

In [ ]:
history_dict = history.history
epochs_range = range(1, len(history_dict["loss"])+1)

plt.plot(epochs_range, history_dict["loss"], label="Train Loss")
plt.plot(epochs_range, history_dict["val_loss"], label="Val Loss")
plt.legend()
plt.show()

plt.plot(epochs_range, history_dict["accuracy"], label="Train Accuracy")
plt.plot(epochs_range, history_dict["val_accuracy"], label="Val Accuracy")
plt.legend()
plt.show()

### D4: Test Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test accuracy:", test_acc)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int).ravel()

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

## Part E – Save the Model

In [ ]:
model.save("Sentiment Analysis model.keras")
